In [1]:
# Level 1 – Task 3: Cleaning Data
### Dataset: Titanic Passenger Dataset
# Objective: Demonstrate professional data cleaning skills by transforming a messy dataset into a clean, analysis-ready dataset. Every decision is documented.

import pandas as pd
import numpy as np

In [2]:
# Load the dataset
df = pd.read_csv('Titanic-Dataset.csv')


print("First 5 rows:")
df.head()

First 5 rows:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
## 2. Data Quality Report (Before Cleaning)

#We start by examining the structure, data types, missing values, and basic statistics to understand the quality issues present in the raw data.
print("Shape of the dataset:", df.shape)
print("\nColumn Data Types:")
print(df.dtypes)
print("\nNull Values per Column:")
print(df.isnull().sum())
print("\nPercentage of Missing Values:")
print((df.isnull().sum() / len(df) * 100).round(2))

Shape of the dataset: (891, 12)

Column Data Types:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

Null Values per Column:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Percentage of Missing Values:
PassengerId     0.00
Survived        0.00
Pclass          0.00
Name            0.00
Sex             0.00
Age            19.87
SibSp           0.00
Parch           0.00
Ticket          0.00
Fare            0.00
Cabin          77.10
Embarked        0.22
dtype: float64


In [4]:
# Descriptive statistics
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [5]:
## 3. Missing Value Handling

# Strategy & Justification:

#Age → Filled with **median**  
  #Reason: Age is skewed (has outliers). Median is more robust than mean.

#Embarked → Filled with **mode** (most frequent value)  
  #Reason: Only 2 missing values. Mode is the best choice for categorical data.

#Cabin → **Dropped** the entire column  
  #Reason: 687 out of 891 values are missing (~77%). Too many missing values to impute meaningfully.
    
# Age → median
df['Age'] = df['Age'].fillna(df['Age'].median())

# Embarked → mode
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Cabin → drop the column (too many missing values)
df = df.drop(columns=['Cabin'])

print("Null Values After Cleaning:")
print(df.isnull().sum())

Null Values After Cleaning:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


In [6]:
## 4. Duplicate Removal
#We check for exact duplicate rows. In this dataset there were **0 duplicates**, so no rows were removed. This step is still important to document.

print("Duplicates before:", df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicates after:", df.duplicated().sum())
print(f"Rows removed: {0}")   # for clarity

Duplicates before: 0
Duplicates after: 0
Rows removed: 0


In [7]:
## 5. Standardisation / Formatting

# Converted `Sex` to lowercase and removed extra spaces.
# Converted `Embarked` to uppercase and removed extra spaces.
# This ensures consistency in categorical values.

df['Sex'] = df['Sex'].str.lower().str.strip()
df['Embarked'] = df['Embarked'].str.upper().str.strip()

print("Sex values:", df['Sex'].unique())
print("Embarked values:", df['Embarked'].unique())

Sex values: ['male' 'female']
Embarked values: ['S' 'C' 'Q']


In [8]:
## 6. Data Type Correction

# `PassengerId` converted to **string** (it is an identifier, not a numerical value for calculation).
# Other columns already have appropriate data types.

df['PassengerId'] = df['PassengerId'].astype(str)

print("Updated Data Types:")
print(df.dtypes)

Updated Data Types:
PassengerId     object
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Embarked        object
dtype: object


In [9]:
## 7. Outlier Detection & Treatment

# We used the **IQR (Interquartile Range)** method to detect outliers in `Age` and `Fare`.

# Decision: We chose to **cap** the outliers (Winsorization) instead of removing them.

# Justification:
# Removing outliers would reduce the dataset size significantly (especially Fare has many high values).
# Capping preserves all rows while reducing the influence of extreme values.
# This is a common and preferred approach in many real-world scenarios.

for col in ['Age', 'Fare']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers)} outliers found (lower={lower:.2f}, upper={upper:.2f})")

Age: 66 outliers found (lower=2.50, upper=54.50)
Fare: 116 outliers found (lower=-26.72, upper=65.63)


In [10]:
for col in ['Age', 'Fare']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower=lower, upper=upper)

print("Outliers capped successfully!")
print("\nNew Age range:", df['Age'].min(), "to", df['Age'].max())
print("New Fare range:", df['Fare'].min(), "to", df['Fare'].max())

Outliers capped successfully!

New Age range: 2.5 to 54.5
New Fare range: 0.0 to 65.6344


In [11]:
## 8. Before vs After Summary

# Comparison of key data quality metrics before and after cleaning.

# Before vs After Summary
summary = {
    'Metric': [
        'Total Rows',
        'Total Columns',
        'Null Values',
        'Duplicate Rows',
        'Cabin Column',
        'PassengerId dtype'
    ],
    'Before Cleaning': [
        891,
        12,
        866,                    # 177 Age + 687 Cabin + 2 Embarked
        0,
        'Present (77% missing)',
        'int64'
    ],
    'After Cleaning': [
        len(df),
        len(df.columns),
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        'Dropped',
        str(df['PassengerId'].dtype)
    ]
}

summary_df = pd.DataFrame(summary)
summary_df

,Metric,Before Cleaning,After Cleaning
0,Total Rows,891,891
1,Total Columns,12,11
2,Null Values,866,0
3,Duplicate Rows,0,0
4,Cabin Column,Present (77% missing),Dropped
5,PassengerId dtype,int64,object


In [12]:
## 9. Save Cleaned Dataset

#The cleaned dataset is saved as `Titanic-Cleaned.csv` for future analysis.

df.to_csv('Titanic-Cleaned.csv', index=False)

print("Cleaned dataset saved successfully as 'Titanic-Cleaned.csv'")
print("Final Shape:", df.shape)
print("\nFinal Preview:")
df.head()

Cleaned dataset saved successfully as 'Titanic-Cleaned.csv'
Final Shape: (891, 11)

Final Preview:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,65.6344,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


In [ ]:
---
### Cleaning Summary

| Step                        | Action Taken                           | Justification                              |
|-----------------------------|----------------------------------------|--------------------------------------------|
| Missing Age                 | Median imputation                      | Robust to outliers                         |
| Missing Embarked            | Mode imputation                        | Only 2 missing values                      |
| Cabin                       | Dropped                                | 77% missing values                         |
| Duplicates                  | Checked (0 found)                      | No action needed                           |
| Sex & Embarked              | Standardised (lower/upper + strip)     | Consistency                                |
| PassengerId                 | Converted to string                    | Identifier, not numeric                    |
| Age & Fare outliers         | Capped using IQR                       | Preserve data while reducing extreme impact|

**Result:** Clean, consistent, analysis-ready dataset with 891 rows and 11 columns.